# SVM

The main notebook already contains an SVM, but only inside the **2D drift-adaptive ensemble**
(Vergara-style: one one-vs-one SVM per batch, weighted pairwise coupling, pre-aging). That scored
**0.7539 / 0.7527** — below the plain RF baseline. What was never tried is a *straight, properly
preprocessed* SVM, and there is a specific reason to expect that to matter here.

**An RBF kernel is acutely scale-sensitive.** It scores similarity as `exp(-gamma * ||x - x'||^2)`,
so whichever feature has the largest range dominates the distance and everything else becomes
noise. Our raw features span five orders of magnitude (`feat_1` ~ 1e5, the EMA transients ~ 1e0).
The 2D ensemble only ever saw MinMax `[-1, 1]` scaling, which squashes the range but leaves the
*distribution* extremely skewed — a handful of huge values push almost every other sample into a
narrow band.

The `signed_log` transform (`sign(x)·log1p(|x|)`) that just took the MLP from 0.7786 to 0.8506 is
aimed at exactly this problem, so this notebook tests preprocessing **before** touching kernels or
hyperparameters — the most likely source of a real gain.

### Rules carried over

- **Forward-chaining only** — train on batches `< n`, validate on `n`. Never trains on the future.
- **Judged on the large-drift folds.** Folds 7 and 9 had tiny drift steps (1.42, 1.44) and inflate
  the mean; the batch 9 → 10 step the submission must survive is **7.22**, resembling folds 2-6, 8.
- **Hyperparameters are selected walk-forward** (chosen on earlier folds, applied to the next),
  with the hindsight-optimal number reported alongside purely to show the gap.

**Reference points, same scheme:** RF 0.7807 / 0.828 large-drift · SVM 2D ensemble 0.7539 ·
OSC k=2 0.8477 / 0.828 · **MLP (signed-log) 0.8506 / 0.8636**.

In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_sub = pd.read_csv("data/sample_submission.csv")

FEAT = [f"feat_{i}" for i in range(1, 129)]
COLS = FEAT + ["concentration"]
CLASSES = sorted(train["gas_class"].unique())
FOLDS = sorted(train["batch"].unique())[1:]

_sc = StandardScaler().fit(train[FEAT])
_X = _sc.transform(train[FEAT])
_c = {b: _X[train["batch"].values == b].mean(0) for b in sorted(train["batch"].unique())}
DRIFT_STEP = {b: float(np.linalg.norm(_c[b] - _c[b - 1])) for b in FOLDS}
LARGE_DRIFT = [b for b, s in DRIFT_STEP.items() if s >= 5.0]
print("drift step per fold:", {int(b): round(s, 2) for b, s in DRIFT_STEP.items()})
print(f"large-drift folds (>=5, like the 7.22 of batch 9->10): {[int(b) for b in LARGE_DRIFT]}")
print(f"\ntrain {train.shape}, test {test.shape}")

drift step per fold: {2: 5.02, 3: 5.55, 4: 9.15, 5: 7.47, 6: 9.59, 7: 1.42, 8: 5.57, 9: 1.44}
large-drift folds (>=5, like the 7.22 of batch 9->10): [2, 3, 4, 5, 6, 8]

train (10310, 132), test (3600, 131)


In [2]:
def signed_log(a):
    return np.sign(a) * np.log1p(np.abs(a))


# Four preprocessing pipelines. Each is fit on the TRAINING fold only.
PREP = {
    "raw + standardise":        lambda: ("std", None),
    "MinMax [-1,1] (2D ens.)":  lambda: ("minmax", None),
    "signed-log + standardise": lambda: ("std", signed_log),
    "quantile -> normal":       lambda: ("quantile", None),
}


def make_prep(kind, pre):
    def build(fit_df, *apply_dfs):
        f = pre if pre is not None else (lambda x: x)
        A = f(fit_df[COLS].values)
        if kind == "std":
            sc = StandardScaler().fit(A)
        elif kind == "minmax":
            sc = MinMaxScaler(feature_range=(-1, 1)).fit(A)
        else:
            sc = QuantileTransformer(output_distribution="normal", n_quantiles=500,
                                     random_state=0).fit(A)
        return [sc.transform(f(d[COLS].values)) for d in (fit_df,) + apply_dfs]
    return build


def eval_svm(build, make_clf, label="", folds=FOLDS, verbose=True):
    """Forward-chaining: train on batches < n, score batch n."""
    out = []
    for vb in folds:
        tr, va = train[train["batch"] < vb], train[train["batch"] == vb]
        Xt, Xv = build(tr, va)
        clf = make_clf().fit(Xt, tr["gas_class"])
        out.append((vb, f1_score(va["gas_class"], clf.predict(Xv), average="macro")))
    d = pd.DataFrame(out, columns=["batch", "f1"])
    m, big = d.f1.mean(), d[d.batch.isin(LARGE_DRIFT)].f1.mean()
    if verbose:
        print(f"  {label:<40} mean={m:.4f}  large-drift={big:.4f}")
    return d, m, big

In [3]:
# Step 1: preprocessing, at fixed sensible hyperparameters. Cheapest, biggest expected effect.
print("=== preprocessing comparison (RBF, C=10, gamma='scale') ===")
t0 = time.time()
prep_res = {}
for name, spec in PREP.items():
    kind, pre = spec()
    d, m, big = eval_svm(make_prep(kind, pre),
                         lambda: SVC(kernel="rbf", C=10, gamma="scale"), label=name)
    prep_res[name] = (m, big, d)
BEST_PREP = max(prep_res, key=lambda k: prep_res[k][1])   # rank on large-drift folds
print(f"\nbest preprocessing (by large-drift): {BEST_PREP}   [{time.time() - t0:.0f}s]")

=== preprocessing comparison (RBF, C=10, gamma='scale') ===


  raw + standardise                        mean=0.8116  large-drift=0.8157


  MinMax [-1,1] (2D ens.)                  mean=0.8058  large-drift=0.8050


  signed-log + standardise                 mean=0.8620  large-drift=0.8585


C:\Users\Pc\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_data.py:2905: UserWarning: n_quantiles (500) is greater than the total number of samples (445). n_quantiles is set to n_samples.
  warnings.warn(


  quantile -> normal                       mean=0.8487  large-drift=0.8362

best preprocessing (by large-drift): signed-log + standardise   [5s]


In [4]:
# Step 2: kernels, on the winning preprocessing.
print(f"=== kernel comparison (preprocessing: {BEST_PREP}) ===")
kind, pre = PREP[BEST_PREP]()
build = make_prep(kind, pre)

t0 = time.time()
kern_res = {}
for kname, mk in {
    "rbf":                 lambda: SVC(kernel="rbf", C=10, gamma="scale"),
    "linear (SVC)":        lambda: SVC(kernel="linear", C=1),
    "poly deg 2":          lambda: SVC(kernel="poly", degree=2, C=10, gamma="scale"),
    "rbf + balanced":      lambda: SVC(kernel="rbf", C=10, gamma="scale", class_weight="balanced"),
}.items():
    _, m, big = eval_svm(build, mk, label=kname)
    kern_res[kname] = (m, big)
print(f"[{time.time() - t0:.0f}s]")

=== kernel comparison (preprocessing: signed-log + standardise) ===


  rbf                                      mean=0.8620  large-drift=0.8585


  linear (SVC)                             mean=0.8221  large-drift=0.8436


  poly deg 2                               mean=0.7376  large-drift=0.7239


  rbf + balanced                           mean=0.8621  large-drift=0.8565
[4s]


In [5]:
# Step 3: C / gamma grid. Compute the full fold x config matrix ONCE, then derive both the
# honest walk-forward choice and the (unobtainable) hindsight-best from the same numbers.
GRID = [(c, g) for c in (1, 10, 100) for g in ("scale", 0.01, 0.1)]
print(f"=== C / gamma grid on {BEST_PREP} ({len(GRID)} configs x {len(FOLDS)} folds) ===")

t0 = time.time()
grid_f1 = {}
for c, g in GRID:
    d, m, big = eval_svm(build, lambda c=c, g=g: SVC(kernel="rbf", C=c, gamma=g),
                         label=f"C={c}, gamma={g}")
    grid_f1[(c, g)] = dict(zip(d.batch, d.f1))
print(f"[{time.time() - t0:.0f}s]")

# walk-forward: for each fold pick the config that was best on the EARLIER folds only
wf = []
for i, vb in enumerate(FOLDS):
    if i == 0:
        cfg = (10, "scale")
    else:
        prev = FOLDS[:i]
        cfg = max(GRID, key=lambda k: np.mean([grid_f1[k][b] for b in prev]))
    wf.append(dict(batch=vb, config=str(cfg), f1=grid_f1[cfg][vb]))
wf = pd.DataFrame(wf)

best_fixed = max(GRID, key=lambda k: np.mean([grid_f1[k][b] for b in LARGE_DRIFT]))
wf_mean, wf_big = wf.f1.mean(), wf[wf.batch.isin(LARGE_DRIFT)].f1.mean()
bf_mean = np.mean([grid_f1[best_fixed][b] for b in FOLDS])
bf_big = np.mean([grid_f1[best_fixed][b] for b in LARGE_DRIFT])

print(f"\nwalk-forward selection : mean={wf_mean:.4f}  large-drift={wf_big:.4f}")
print(f"hindsight-best {str(best_fixed):<16}: mean={bf_mean:.4f}  large-drift={bf_big:.4f}  (optimistic)")
print("\nper-fold config actually chosen:")
print(wf.to_string(index=False))

=== C / gamma grid on signed-log + standardise (9 configs x 8 folds) ===


  C=1, gamma=scale                         mean=0.8092  large-drift=0.8226


  C=1, gamma=0.01                          mean=0.8131  large-drift=0.8273


  C=1, gamma=0.1                           mean=0.7425  large-drift=0.7604


  C=10, gamma=scale                        mean=0.8620  large-drift=0.8585


  C=10, gamma=0.01                         mean=0.8584  large-drift=0.8549


  C=10, gamma=0.1                          mean=0.7442  large-drift=0.7567


  C=100, gamma=scale                       mean=0.8487  large-drift=0.8387


  C=100, gamma=0.01                        mean=0.8450  large-drift=0.8396


  C=100, gamma=0.1                         mean=0.7442  large-drift=0.7567
[15s]

walk-forward selection : mean=0.8620  large-drift=0.8585
hindsight-best (10, 'scale')   : mean=0.8620  large-drift=0.8585  (optimistic)

per-fold config actually chosen:
 batch        config       f1
     2 (10, 'scale') 0.727644
     3 (10, 'scale') 0.981105
     4 (10, 'scale') 0.888212
     5 (10, 'scale') 0.980656
     6 (10, 'scale') 0.619715
     7 (10, 'scale') 0.797910
     8 (10, 'scale') 0.953827
     9 (10, 'scale') 0.946967


In [6]:
rows = [dict(model=f"SVM rbf, {k}", mean=v[0], large_drift=v[1]) for k, v in prep_res.items()]
rows += [dict(model=f"SVM {k} ({BEST_PREP})", mean=v[0], large_drift=v[1]) for k, v in kern_res.items()]
rows += [dict(model="SVM tuned, walk-forward", mean=wf_mean, large_drift=wf_big),
         dict(model=f"SVM tuned, hindsight {best_fixed}", mean=bf_mean, large_drift=bf_big)]
rows += [dict(model="-- SVM 2D ensemble (main nb)", mean=0.7539, large_drift=float("nan")),
         dict(model="-- RF baseline (main nb)", mean=0.7807, large_drift=0.828),
         dict(model="-- OSC k=2 (main nb)", mean=0.8477, large_drift=0.828),
         dict(model="-- MLP signed-log (ensemble nb)", mean=0.8506, large_drift=0.8636)]
summary = pd.DataFrame(rows).sort_values("large_drift", ascending=False, na_position="last")

print("=" * 82)
print(f"{'model':<44}{'mean F1':>11}{'large-drift':>14}")
print("-" * 82)
for _, r in summary.iterrows():
    ld = "     n/a" if pd.isna(r.large_drift) else f"{r.large_drift:>14.4f}"
    print(f"{r.model:<44}{r['mean']:>11.4f}{ld}")
print("=" * 82)
print("rows prefixed '--' are reference points from the other notebooks, same forward-chaining scheme")

model                                           mean F1   large-drift
----------------------------------------------------------------------------------
-- MLP signed-log (ensemble nb)                  0.8506        0.8636
SVM rbf (signed-log + standardise)               0.8620        0.8585
SVM tuned, hindsight (10, 'scale')               0.8620        0.8585
SVM tuned, walk-forward                          0.8620        0.8585
SVM rbf, signed-log + standardise                0.8620        0.8585
SVM rbf + balanced (signed-log + standardise)     0.8621        0.8565
SVM linear (SVC) (signed-log + standardise)      0.8221        0.8436
SVM rbf, quantile -> normal                      0.8487        0.8362
-- RF baseline (main nb)                         0.7807        0.8280
-- OSC k=2 (main nb)                             0.8477        0.8280
SVM rbf, raw + standardise                       0.8116        0.8157
SVM rbf, MinMax [-1,1] (2D ens.)                 0.8058        0.8050
SVM po

In [7]:
# Refit on all 9 batches and predict batch 10. Uses the WALK-FORWARD config (the last one
# chosen from real history), not the hindsight-best, and writes a separate file.
final_cfg = eval(wf.iloc[-1]["config"])
print(f"final config: C={final_cfg[0]}, gamma={final_cfg[1]}, preprocessing={BEST_PREP}")

X_all, X_te = build(train, test)
clf = SVC(kernel="rbf", C=final_cfg[0], gamma=final_cfg[1]).fit(X_all, train["gas_class"])
pred = clf.predict(X_te)

sub = pd.DataFrame({"measurement_id": test["measurement_id"], "gas_class": pred})
assert list(sub.columns) == list(sample_sub.columns)
assert len(sub) == len(sample_sub)
assert (sub["measurement_id"].values == sample_sub["measurement_id"].values).all()
assert sub["gas_class"].isin(range(1, 7)).all() and sub["gas_class"].notna().all()
sub.to_csv("data/submission_svm.csv", index=False)
print("wrote data/submission_svm.csv  (data/submission.csv left untouched)")

vc = sub["gas_class"].value_counts().sort_index()
print("\npredicted distribution vs the 600/class the competition states:")
for c in CLASSES:
    print(f"  class {c}: {vc.get(c, 0):>5}  ({vc.get(c, 0) - 600:+d})")
print(f"  total absolute deviation: {int((vc.reindex(CLASSES).fillna(0) - 600).abs().sum())}")

# Support-vector fraction is a sanity read on the fitted model: near 100% would mean the kernel
# cannot separate the classes and the SVM is effectively memorising; a small fraction means a
# sparse, well-separated solution.
sv_frac = clf.n_support_.sum() / len(X_all)
print(f"\nsupport vectors: {clf.n_support_.sum():,} of {len(X_all):,} training rows ({sv_frac:.1%})")
print("  -> sparse solution: the signed-log space separates the classes cleanly, so the decision")
print("     boundary rests on a small subset of samples rather than on nearly all of them.")
print("     (this is the healthy case; a fraction near 100% would indicate memorisation)")

final config: C=10, gamma=scale, preprocessing=signed-log + standardise


wrote data/submission_svm.csv  (data/submission.csv left untouched)

predicted distribution vs the 600/class the competition states:
  class 1:   877  (+277)
  class 2:   609  (+9)
  class 3:   524  (-76)
  class 4:   391  (-209)
  class 5:   451  (-149)
  class 6:   748  (+148)
  total absolute deviation: 868

support vectors: 661 of 10,310 training rows (6.4%)
  -> sparse solution: the signed-log space separates the classes cleanly, so the decision
     boundary rests on a small subset of samples rather than on nearly all of them.
     (this is the healthy case; a fraction near 100% would indicate memorisation)
